In [167]:
import os
import pandas as pd
import numpy as np

In [168]:
BASE_DIR = "dataset"

dfs = {}

for file in os.listdir(BASE_DIR):
    if file.endswith(".csv"):
        path = os.path.join(BASE_DIR, file)

        df = pd.read_csv(path)

        name = file.replace(".csv", "")
        dfs[name] = df
dfs.keys()
tables_name = dfs.keys()-{'data_dictionary'}

>> Data Inspection

In [169]:
table_info = []
for name in tables_name:
    data = dfs[name]
    table_info.append(
        {"Table":name,
         "Rows":data.shape[0],
         "Columns":data.shape[1],
         "missing_value":data.isna().sum().sum(),
         "duplicated data":data.duplicated().sum()
         }
    )
display(pd.DataFrame(table_info))
    

,Table,Rows,Columns,missing_value,duplicated data
0,products,7,3,0,0
1,sales_pipeline,8800,8,6103,0
2,sales_teams,35,3,0,0
3,accounts,85,7,70,0


In [170]:
table_columns_type = []
for name in tables_name:
    data = dfs[name].dtypes
    for column,type in data.items():
        table_columns_type.append(
            {"table":name,
            "column":column,
            "type of data":type,
            "missing values": dfs[name][column].isna().sum(),
            "unique Values": dfs[name][column].nunique()}
        )
data_table_info = pd.DataFrame(table_columns_type)


>> Data Cleaning

In [171]:
df_accounts = dfs['accounts']
display(data_table_info[data_table_info['table']=="accounts"])
df_accounts.loc[df_accounts['subsidiary_of'].isna(),'subsidiary_of'] = 'Unknown'
columns = (df_accounts.select_dtypes(include=['object','str'])).columns
for column in columns:
    df_accounts[column] = df_accounts[column].str.strip()
display(df_accounts)
display(df_accounts.info())


,table,column,type of data,missing values,unique Values
14,accounts,account,str,0,85
15,accounts,sector,str,0,10
16,accounts,year_established,int64,0,35
17,accounts,revenue,float64,0,85
18,accounts,employees,int64,0,85
19,accounts,office_location,str,0,15
20,accounts,subsidiary_of,str,70,7


,account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,Acme Corporation,technolgy,1996,1100.04,2822,United States,Unknown
1,Betasoloin,medical,1999,251.41,495,United States,Unknown
2,Betatech,medical,1986,647.18,1185,Kenya,Unknown
3,Bioholding,medical,2012,587.34,1356,Philipines,Unknown
4,Bioplex,medical,1991,326.82,1016,United States,Unknown
...,...,...,...,...,...,...,...
80,Zathunicon,retail,2010,71.12,144,United States,Unknown
81,Zencorporation,technolgy,2011,40.79,142,China,Unknown
82,Zoomit,entertainment,1992,324.19,978,United States,Unknown
83,Zotware,software,1979,4478.47,13809,United States,Unknown


<class 'pandas.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   account           85 non-null     str    
 1   sector            85 non-null     str    
 2   year_established  85 non-null     int64  
 3   revenue           85 non-null     float64
 4   employees         85 non-null     int64  
 5   office_location   85 non-null     str    
 6   subsidiary_of     85 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 4.8 KB


None

In [172]:
df_products = dfs['products']
display(data_table_info[data_table_info['table']=='products'])
columns = (df_products.select_dtypes(include = 'str')).columns
for column in columns:
    df_products[column] = df_products[column].str.strip()
display(df_products)

,table,column,type of data,missing values,unique Values
0,products,product,str,0,7
1,products,series,str,0,3
2,products,sales_price,int64,0,7


,product,series,sales_price
0,GTX Basic,GTX,550
1,GTX Pro,GTX,4821
2,MG Special,MG,55
3,MG Advanced,MG,3393
4,GTX Plus Pro,GTX,5482
5,GTX Plus Basic,GTX,1096
6,GTK 500,GTK,26768


In [173]:
df_sales_pipeline = dfs['sales_pipeline']
display(data_table_info[data_table_info['table']=='sales_pipeline'])
df_sales_pipeline.loc[df_sales_pipeline['account'].isna(),'account'] = 'Unidentified'
column_str = df_sales_pipeline.select_dtypes(
    include=['object', 'str']
).columns

for column in column_str:
    df_sales_pipeline[column] = df_sales_pipeline[column].str.strip()

display(pd.crosstab(
    df_sales_pipeline['deal_stage'],
    df_sales_pipeline['engage_date'].isna()
))
display(pd.crosstab(
    df_sales_pipeline['deal_stage'],
    df_sales_pipeline['close_date'].isna()
))


,table,column,type of data,missing values,unique Values
3,sales_pipeline,opportunity_id,str,0,8800
4,sales_pipeline,sales_agent,str,0,30
5,sales_pipeline,product,str,0,7
6,sales_pipeline,account,str,1425,85
7,sales_pipeline,deal_stage,str,0,4
8,sales_pipeline,engage_date,str,500,421
9,sales_pipeline,close_date,str,2089,306
10,sales_pipeline,close_value,float64,2089,2051


engage_date,False,True
deal_stage,,
Engaging,1589,0
Lost,2473,0
Prospecting,0,500
Won,4238,0


close_date,False,True
deal_stage,,
Engaging,0,1589
Lost,2473,0
Prospecting,0,500
Won,4238,0


In [ ]:
df_sales_pipeline_Prospecting = df_sales_pipeline[df_sales_pipeline['deal_stage']=='Prospecting'].reset_index(drop=True) 

df_sales_pipeline_Lost = df_sales_pipeline[df_sales_pipeline['deal_stage']=='Lost'].reset_index(drop=True)

df_sales_pipeline_Won = df_sales_pipeline[df_sales_pipeline['deal_stage']=='Won'].reset_index(drop=True)

date_column= ['engage_date','close_date']
for column in date_column:
    df_sales_pipeline_Lost[column] = pd.to_datetime(df_sales_pipeline_Lost[column])
    df_sales_pipeline_Won[column] = pd.to_datetime(df_sales_pipeline_Won[column])  
display(df_sales_pipeline_Won.info())
display(df_sales_pipeline_Lost.info())



<class 'pandas.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   opportunity_id  4238 non-null   str           
 1   sales_agent     4238 non-null   str           
 2   product         4238 non-null   str           
 3   account         4238 non-null   str           
 4   deal_stage      4238 non-null   str           
 5   engage_date     4238 non-null   datetime64[us]
 6   close_date      4238 non-null   datetime64[us]
 7   close_value     4238 non-null   float64       
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 265.0 KB


None

<class 'pandas.DataFrame'>
RangeIndex: 2473 entries, 0 to 2472
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   opportunity_id  2473 non-null   str           
 1   sales_agent     2473 non-null   str           
 2   product         2473 non-null   str           
 3   account         2473 non-null   str           
 4   deal_stage      2473 non-null   str           
 5   engage_date     2473 non-null   datetime64[us]
 6   close_date      2473 non-null   datetime64[us]
 7   close_value     2473 non-null   float64       
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 154.7 KB


None

In [185]:
df_sales_teams = dfs['sales_teams']
display(data_table_info[data_table_info['table']=="sales_teams"])
columns = df_sales_teams.columns
for column in columns:
    df_sales_teams[column] = df_sales_teams[column].str.strip()
display(df_sales_teams.head())
display(df_sales_teams.info())


,table,column,type of data,missing values,unique Values
11,sales_teams,sales_agent,str,0,35
12,sales_teams,manager,str,0,6
13,sales_teams,regional_office,str,0,3


,sales_agent,manager,regional_office
0,Anna Snelling,Dustin Brinkmann,Central
1,Cecily Lampkin,Dustin Brinkmann,Central
2,Versie Hillebrand,Dustin Brinkmann,Central
3,Lajuana Vencill,Dustin Brinkmann,Central
4,Moses Frase,Dustin Brinkmann,Central


<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sales_agent      35 non-null     str  
 1   manager          35 non-null     str  
 2   regional_office  35 non-null     str  
dtypes: str(3)
memory usage: 972.0 bytes


None

>>Data Exploratory

In [187]:
df_accounts.describe()


,year_established,revenue,employees
count,85.000000,85.000000,85.000000
mean,1996.105882,1994.632941,4660.823529
std,8.865427,2169.491436,5715.601198
min,1979.000000,4.540000,9.000000
25%,1989.000000,497.110000,1179.000000
50%,1996.000000,1223.720000,2769.000000
75%,2002.000000,2741.370000,5595.000000
max,2017.000000,11698.030000,34288.000000


In [191]:
df_sales_teams.describe()

,sales_agent,manager,regional_office
count,35,35,35
unique,35,6,3
top,Anna Snelling,Melvin Marxen,East
freq,1,6,12


In [194]:
df_sales_pipeline.describe()

,close_value
count,6711.000000
mean,1490.915512
std,2320.670773
min,0.000000
25%,0.000000
50%,472.000000
75%,3225.000000
max,30288.000000


In [196]:
df_sales_pipeline_Prospecting.describe()

,close_value
count,0.0
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [ ]:
df_sales_pipeline_Lost.describe()

,engage_date,close_date,close_value
count,2473,2473,2473.0
mean,2017-06-22 02:11:00.897695,2017-08-02 13:41:01.625556,0.0
min,2016-11-04 00:00:00,2017-03-01 00:00:00,0.0
25%,2017-04-12 00:00:00,2017-05-19 00:00:00,0.0
50%,2017-06-29 00:00:00,2017-08-02 00:00:00,0.0
75%,2017-09-08 00:00:00,2017-10-19 00:00:00,0.0
max,2017-12-22 00:00:00,2017-12-31 00:00:00,0.0
std,NaN,NaN,0.0


In [197]:
df_sales_pipeline_Won.describe()

,engage_date,close_date,close_value
count,4238,4238,4238.000000
mean,2017-06-09 12:50:58.046248,2017-07-31 07:37:00.481359,2360.909391
min,2016-10-20 00:00:00,2017-03-01 00:00:00,38.000000
25%,2017-03-25 00:00:00,2017-05-18 00:00:00,518.000000
50%,2017-06-12 00:00:00,2017-08-02 00:00:00,1117.000000
75%,2017-09-01 00:00:00,2017-10-16 00:00:00,4429.500000
max,2017-12-27 00:00:00,2017-12-31 00:00:00,30288.000000
std,NaN,NaN,2544.484463


In [198]:
df_products.describe()

,sales_price
count,7.000000
mean,6023.571429
std,9388.428070
min,55.000000
25%,823.000000
50%,3393.000000
75%,5151.500000
max,26768.000000
